In [2]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sub = pd.read_csv("sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
train.head()

Train: (8693, 14)
Test: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
print("--- Información de train ---")
train.info()

print("\n--- Valores nulos en train ---")
print(train.isnull().sum()[train.isnull().sum() > 0])

print("\n--- Valores nulos en test ---")
print(test.isnull().sum()[test.isnull().sum() > 0])

print("\n--- Distribución de Transported ---")
print(train["Transported"].value_counts(dropna=False))

--- Información de train ---
<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB

--- Valores nulos en train ---
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age 

In [4]:
import numpy as np

spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

def construir_features(df):
    data = df.copy()
    
    # Grupo y tamaño del grupo a partir de PassengerId (gggg_pp)
    grupo_split = data["PassengerId"].str.split("_", expand=True)
    data["Group"] = grupo_split[0]
    data["GroupSize"] = data.groupby("Group")["Group"].transform("count")
    
    # Cubierta, número y lado a partir de Cabin (deck/num/side)
    cabin_split = data["Cabin"].str.split("/", expand=True)
    data["Deck"] = cabin_split[0]
    data["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
    data["Side"] = cabin_split[2]
    
    # Gasto total en servicios y bandera de si no gastó nada
    data["TotalSpend"] = data[spend_cols].fillna(0).sum(axis=1)
    data["SinGasto"] = (data["TotalSpend"] == 0).astype(int)
    
    return data

train_prep = construir_features(train)
test_prep = construir_features(test)

train_prep[["PassengerId", "Group", "GroupSize", "Deck", "CabinNum", "Side", "TotalSpend", "SinGasto"]].head()

,PassengerId,Group,GroupSize,Deck,CabinNum,Side,TotalSpend,SinGasto
0,0001_01,0001,1,B,0.0,P,0.0,1
1,0002_01,0002,1,F,0.0,S,736.0,0
2,0003_01,0003,2,A,0.0,S,10383.0,0
3,0003_02,0003,2,A,0.0,S,5176.0,0
4,0004_01,0004,1,F,1.0,S,1091.0,0


In [5]:
import numpy as np

spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

def construir_features(df):
    data = df.copy()
    
    # Grupo y tamaño del grupo a partir de PassengerId (gggg_pp)
    grupo_split = data["PassengerId"].str.split("_", expand=True)
    data["Group"] = grupo_split[0]
    data["GroupSize"] = data.groupby("Group")["Group"].transform("count")
    
    # Cubierta, número y lado a partir de Cabin (deck/num/side)
    cabin_split = data["Cabin"].str.split("/", expand=True)
    data["Deck"] = cabin_split[0]
    data["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
    data["Side"] = cabin_split[2]
    
    # Gasto total en servicios y bandera de si no gastó nada
    data["TotalSpend"] = data[spend_cols].fillna(0).sum(axis=1)
    data["SinGasto"] = (data["TotalSpend"] == 0).astype(int)
    
    return data

train_prep = construir_features(train)
test_prep = construir_features(test)

train_prep[["PassengerId", "Group", "GroupSize", "Deck", "CabinNum", "Side", "TotalSpend", "SinGasto"]].head()

,PassengerId,Group,GroupSize,Deck,CabinNum,Side,TotalSpend,SinGasto
0,0001_01,0001,1,B,0.0,P,0.0,1
1,0002_01,0002,1,F,0.0,S,736.0,0
2,0003_01,0003,2,A,0.0,S,10383.0,0
3,0003_02,0003,2,A,0.0,S,5176.0,0
4,0004_01,0004,1,F,1.0,S,1091.0,0


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

target = "Transported"
cols_a_excluir = ["PassengerId", "Name", "Cabin", "Group", target]
features = [col for col in train_prep.columns if col not in cols_a_excluir]

X = train_prep[features]
y = train_prep[target].astype(int)

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [col for col in X.columns if col not in cat_cols]

num_transform = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_transform = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocesador = ColumnTransformer([
    ("num", num_transform, num_cols),
    ("cat", cat_transform, cat_cols)
])

modelo = Pipeline([
    ("prep", preprocesador),
    ("clf", HistGradientBoostingClassifier(random_state=42))
])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelo.fit(X_train, y_train)
pred_val = modelo.predict(X_val)

print(f"Accuracy en validación: {accuracy_score(y_val, pred_val):.4f}\n")
print(classification_report(y_val, pred_val, target_names=["False", "True"]))

C:\Users\rudol\AppData\Local\Temp\ipykernel_14892\3703533326.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


Accuracy en validación: 0.8206

              precision    recall  f1-score   support

       False       0.82      0.82      0.82       863
        True       0.82      0.82      0.82       876

    accuracy                           0.82      1739
   macro avg       0.82      0.82      0.82      1739
weighted avg       0.82      0.82      0.82      1739



In [7]:
modelo.fit(X, y)
pred_test = modelo.predict(test_prep[features]).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": pred_test
})

submission.to_csv("submission.csv", index=False)
print("Archivo submission.csv guardado. Filas:", len(submission))
submission.head()

Archivo submission.csv guardado. Filas: 4277


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
